In [1]:
import pandas as pd 

In [2]:
df=pd.read_csv(r"D:\Kaggle contests\neurips-open-polymer-prediction-2025\data\train.csv")

In [3]:
df.head()

,id,SMILES,Tg,FFV,Tc,Density,Rg
0,87817,*CC(*)c1ccccc1C(=O)OCCCCCC,NaN,0.374645,0.205667,NaN,NaN
1,106919,*Nc1ccc([C@H](CCC)c2ccc(C3(c4ccc([C@@H](CCC)c5...,NaN,0.370410,NaN,NaN,NaN
2,388772,*Oc1ccc(S(=O)(=O)c2ccc(Oc3ccc(C4(c5ccc(Oc6ccc(...,NaN,0.378860,NaN,NaN,NaN
3,519416,*Nc1ccc(-c2c(-c3ccc(C)cc3)c(-c3ccc(C)cc3)c(N*)...,NaN,0.387324,NaN,NaN,NaN
4,539187,*Oc1ccc(OC(=O)c2cc(OCCCCCCCCCOCC3CCCN3c3ccc([N...,NaN,0.355470,NaN,NaN,NaN


In [4]:
df.shape

(7973, 7)

In [5]:
df.isnull().sum()

id            0
SMILES        0
Tg         7462
FFV         943
Tc         7236
Density    7360
Rg         7359
dtype: int64

In [6]:
#!pip install rdkit-pypi

In [7]:
# from rdkit import Chem
# from rdkit.Chem import Draw

# # Example: Polyethylene (replace with your polymer's SMILES)
# smiles = "*CC(*)c1ccccc1C(=O)OCCCCCC"  # Replace with your polymer's SMILES string

# # Generate molecule from SMILES
# mol = Chem.MolFromSmiles(smiles)

# # Generate 2D coordinates
# from rdkit.Chem import AllChem
# AllChem.Compute2DCoords(mol)

# # Draw and save the image
# Draw.MolToFile(mol, 'polymer_structure.png', size=(400, 200))


In [9]:
# from rdkit import Chem
# from rdkit.Chem import Descriptors

# print("Molecular weight:", Descriptors.MolWt(mol))
# print("Number of hydrogen bond donors:", Descriptors.NumHDonors(mol))
# print("Number of hydrogen bond acceptors:", Descriptors.NumHAcceptors(mol))


In [10]:
# from rdkit.Chem import AllChem

# fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
# print(fp.ToList())


In [11]:
# from rdkit.Chem import MolFromSmarts

# query = MolFromSmarts("C=O")  # Example: carbonyl group
# matches = mol.GetSubstructMatches(query)
# print("Number of matches:", len(matches))


In [12]:
# from rdkit.Chem import AssignStereochemistry

# AssignStereochemistry(mol, force=True)
# # After assignment, you can access atom/bond stereo properties


In [13]:
# mol = Chem.AddHs(mol)  # Add hydrogens for 3D conformers
# AllChem.EmbedMolecule(mol)
# AllChem.UFFOptimizeMolecule(mol)


In [26]:
df.describe().loc[['min', 'max']]

,id,Tg,FFV,Tc,Density,Rg
min,8.781700e+04,-148.029738,0.226992,0.0465,0.748691,9.728355
max,2.147438e+09,472.250000,0.777097,0.5240,1.840999,34.672906


In [27]:
import numpy as np
import pandas as pd
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_val_score

class WeightedMAEScorer:
    def __init__(self, property_ranges=None, sample_counts=None):
        """
        Custom scorer for the polymer competition weighted MAE metric
        
        Parameters:
        property_ranges: dict with property names as keys and (min, max) tuples as values
        sample_counts: dict with property names as keys and sample counts as values
        """
        # Default values based on your data
        if property_ranges is None:
            self.property_ranges = {
                'Tg': (-148.029738, 472.250000),
                'FFV': (0.226992, 0.777097),
                'Tc': (0.0465, 0.5240),
                'Density': (0.748691, 1.840999),
                'Rg': (9.728355, 34.672906)
            }
        else:
            self.property_ranges = property_ranges
            
        if sample_counts is None:
            self.sample_counts = {
                'Tg': 2538,
                'FFV': 9057,
                'Tc': 2764,
                'Density': 2640,
                'Rg': 2641
            }
        else:
            self.sample_counts = sample_counts
            
        # Calculate weights
        self.weights = self._calculate_weights()
        
    def _calculate_weights(self):
        """Calculate the competition weights for each property"""
        K = len(self.property_ranges)  # Total number of tasks (5)
        weights = {}
        
        # First calculate individual components
        individual_weights = {}
        for prop in self.property_ranges.keys():
            n_i = self.sample_counts[prop]
            r_i = self.property_ranges[prop][1] - self.property_ranges[prop][0]
            
            # Individual weight components
            freq_weight = 1 / n_i  # Inverse frequency
            scale_weight = 1 / r_i  # Inverse range (scale normalization)
            sqrt_scaling = np.sqrt(1 / n_i)  # Square root of inverse frequency
            
            individual_weights[prop] = {
                'freq_weight': freq_weight,
                'scale_weight': scale_weight, 
                'sqrt_scaling': sqrt_scaling,
                'combined': (freq_weight * scale_weight) / sqrt_scaling
            }
        
        # Calculate normalization factor
        sum_sqrt_inv_n = sum(np.sqrt(1/self.sample_counts[prop]) for prop in self.property_ranges.keys())
        
        # Final weights
        for prop in self.property_ranges.keys():
            n_i = self.sample_counts[prop]
            r_i = self.property_ranges[prop][1] - self.property_ranges[prop][0]
            
            w_i = (1/r_i) * (K * np.sqrt(1/n_i) / sum_sqrt_inv_n)
            weights[prop] = w_i
            
        return weights
    
    def print_weight_analysis(self):
        """Print detailed weight analysis"""
        print("=== Weight Analysis ===")
        print(f"{'Property':<10} {'Samples':<8} {'Range':<12} {'Weight':<12} {'Relative':<10}")
        print("-" * 65)
        
        total_weight = sum(self.weights.values())
        for prop in ['Tg', 'FFV', 'Tc', 'Density', 'Rg']:
            samples = self.sample_counts[prop]
            prop_range = self.property_ranges[prop][1] - self.property_ranges[prop][0]
            weight = self.weights[prop]
            relative = weight / total_weight * 100
            
            print(f"{prop:<10} {samples:<8} {prop_range:<12.4f} {weight:<12.4f} {relative:<10.1f}%")
        
        print(f"\nTotal weight sum: {total_weight:.4f}")
        print(f"Average weight: {total_weight/5:.4f}")
        
    def score(self, y_true, y_pred, return_individual=False):
        """
        Calculate weighted MAE score
        
        Parameters:
        y_true: DataFrame or dict with true values for each property
        y_pred: DataFrame or dict with predicted values for each property
        return_individual: if True, return individual property scores as well
        
        Returns:
        Weighted MAE score (lower is better)
        """
        if isinstance(y_true, pd.DataFrame):
            y_true = y_true.to_dict('series')
        if isinstance(y_pred, pd.DataFrame):
            y_pred = y_pred.to_dict('series')
            
        total_weighted_error = 0
        total_weight = 0
        individual_scores = {}
        
        for prop in self.weights.keys():
            if prop in y_true and prop in y_pred:
                # Get non-null mask
                mask = pd.notna(y_true[prop]) & pd.notna(y_pred[prop])
                
                if mask.sum() > 0:  # If we have valid pairs
                    true_vals = y_true[prop][mask]
                    pred_vals = y_pred[prop][mask]
                    
                    # Calculate MAE for this property
                    mae = np.mean(np.abs(true_vals - pred_vals))
                    
                    # Apply weight
                    weight = self.weights[prop]
                    weighted_error = weight * mae
                    
                    total_weighted_error += weighted_error
                    total_weight += weight
                    
                    individual_scores[prop] = {
                        'mae': mae,
                        'weight': weight,
                        'weighted_mae': weighted_error,
                        'n_samples': mask.sum()
                    }
        
        final_score = total_weighted_error / len(self.weights) if len(self.weights) > 0 else 0
        
        if return_individual:
            return final_score, individual_scores
        return final_score

def create_sklearn_scorer(property_ranges=None, sample_counts=None):
    """
    Create sklearn-compatible scorer for use with cross_val_score
    
    Usage:
    scorer = create_sklearn_scorer()
    scores = cross_val_score(model, X, y, scoring=scorer, cv=5)
    """
    wmae_scorer = WeightedMAEScorer(property_ranges, sample_counts)
    
    def sklearn_wmae_score(estimator, X, y):
        y_pred = estimator.predict(X)
        
        # Convert to proper format if needed
        if hasattr(y, 'columns'):
            y_true_dict = {col: y[col] for col in y.columns}
            if len(y_pred.shape) == 2:
                y_pred_dict = {col: y_pred[:, i] for i, col in enumerate(y.columns)}
            else:
                y_pred_dict = {'target': y_pred}
        else:
            y_true_dict = {'target': y}
            y_pred_dict = {'target': y_pred}
            
        return -wmae_scorer.score(y_true_dict, y_pred_dict)  # Negative because sklearn maximizes
    
    return make_scorer(sklearn_wmae_score, greater_is_better=True)

# Example usage
if __name__ == "__main__":
    # Initialize scorer
    scorer = WeightedMAEScorer()
    
    # Print weight analysis
    scorer.print_weight_analysis()
    
    # Example with dummy data
    n_samples = 100
    np.random.seed(42)
    
    # Create dummy true and predicted values
    y_true = pd.DataFrame({
        'Tg': np.random.normal(100, 50, n_samples),
        'FFV': np.random.normal(0.5, 0.1, n_samples),
        'Tc': np.random.normal(0.3, 0.05, n_samples),
        'Density': np.random.normal(1.2, 0.2, n_samples),
        'Rg': np.random.normal(20, 5, n_samples)
    })
    
    # Add some noise for predictions
    y_pred = y_true + pd.DataFrame({
        'Tg': np.random.normal(0, 10, n_samples),
        'FFV': np.random.normal(0, 0.02, n_samples),
        'Tc': np.random.normal(0, 0.01, n_samples),
        'Density': np.random.normal(0, 0.05, n_samples),
        'Rg': np.random.normal(0, 2, n_samples)
    })
    
    # Calculate score
    final_score, individual_scores = scorer.score(y_true, y_pred, return_individual=True)
    
    print(f"\n=== Example Scoring ===")
    print(f"Overall Weighted MAE: {final_score:.6f}")
    print(f"\nIndividual Property Scores:")
    for prop, scores in individual_scores.items():
        print(f"{prop}: MAE={scores['mae']:.4f}, Weight={scores['weight']:.4f}, "
              f"Weighted={scores['weighted_mae']:.4f}")

=== Weight Analysis ===
Property   Samples  Range        Weight       Relative  
-----------------------------------------------------------------
Tg         2538     620.2797     0.0018       0.0       %
FFV        9057     0.5501       1.0816       24.6      %
Tc         2764     0.4775       2.2556       51.4      %
Density    2640     1.0923       1.0089       23.0      %
Rg         2641     24.9446      0.0442       1.0       %

Total weight sum: 4.3922
Average weight: 0.8784

=== Example Scoring ===
Overall Weighted MAE: 0.031150

Individual Property Scores:
Tg: MAE=7.3881, Weight=0.0018, Weighted=0.0134
FFV: MAE=0.0169, Weight=1.0816, Weighted=0.0183
Tc: MAE=0.0077, Weight=2.2556, Weighted=0.0174
Density: MAE=0.0391, Weight=1.0089, Weighted=0.0395
Rg: MAE=1.5214, Weight=0.0442, Weighted=0.0672


In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Install required packages (uncomment if needed)
# !pip install rdkit-pypi
# !pip install lightgbm

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors
    RDKIT_AVAILABLE = True
except ImportError:
    print("RDKit not available. Install with: pip install rdkit-pypi")
    RDKIT_AVAILABLE = False

import lightgbm as lgb

# ===========================
# 1. DATA LOADING & EDA
# ===========================

def load_and_explore_data(train_path, test_path=None):
    """Load data and perform initial exploration"""
    print("=== LOADING DATA ===")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path) if test_path else None
    
    print(f"Train shape: {train_df.shape}")
    if test_df is not None:
        print(f"Test shape: {test_df.shape}")
    
    print("\n=== MISSING VALUES ===")
    missing = train_df.isnull().sum()
    missing_pct = (missing / len(train_df)) * 100
    missing_df = pd.DataFrame({
        'Missing': missing,
        'Percentage': missing_pct
    }).sort_values('Missing', ascending=False)
    print(missing_df)
    
    # Property statistics
    properties = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
    print("\n=== PROPERTY STATISTICS ===")
    print(train_df[properties].describe())
    
    return train_df, test_df

def visualize_data(train_df):
    """Create visualizations for EDA"""
    properties = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
    
    # Distribution plots
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()
    
    for i, prop in enumerate(properties):
        train_df[prop].hist(bins=50, ax=axes[i], alpha=0.7)
        axes[i].set_title(f'{prop} Distribution')
        axes[i].set_xlabel(prop)
        axes[i].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
    
    # Correlation matrix
    plt.figure(figsize=(8, 6))
    correlation_matrix = train_df[properties].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
    plt.title('Property Correlations')
    plt.show()
    
    return correlation_matrix

# ===========================
# 2. MOLECULAR FEATURE EXTRACTION
# ===========================

def extract_molecular_features(smiles_list):
    """Extract molecular features from SMILES strings"""
    if not RDKIT_AVAILABLE:
        print("RDKit not available. Using dummy features.")
        return create_dummy_features(smiles_list)
    
    features = []
    feature_names = []
    
    print("Extracting molecular features...")
    
    for i, smiles in enumerate(smiles_list):
        if i % 1000 == 0:
            print(f"Processed {i}/{len(smiles_list)} molecules")
            
        mol = Chem.MolFromSmiles(smiles)
        
        if mol is None:
            # Handle invalid SMILES
            features.append([0] * 200)  # Placeholder with zeros
            continue
            
        mol_features = []
        
        # Basic descriptors - using only stable descriptors
        mol_features.extend([
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            Descriptors.NumHDonors(mol),
            Descriptors.NumHAcceptors(mol),
            Descriptors.NumRotatableBonds(mol),
            Descriptors.NumAromaticRings(mol),
            Descriptors.NumAliphaticRings(mol),
            Descriptors.RingCount(mol),
            mol.GetNumAtoms(),  # Simple atom count instead
            Descriptors.NumHeteroatoms(mol),
        ])
        
        # Morgan fingerprints (circular fingerprints)
        morgan_fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, 2, nBits=128)
        mol_features.extend(list(morgan_fp))
        
        # Core molecular descriptors - only using universally available ones
        basic_descriptors = []
        try:
            basic_descriptors = [
                Descriptors.TPSA(mol),
                Descriptors.LabuteASA(mol),
                Descriptors.BertzCT(mol),
                Descriptors.NumRadicalElectrons(mol),
                Descriptors.NumValenceElectrons(mol),
                Descriptors.NumSaturatedRings(mol),
                Descriptors.NumAromaticCarbocycles(mol),
                Descriptors.NumAromaticHeterocycles(mol),
                mol.GetNumBonds(),
                mol.GetNumHeavyAtoms(),
            ]
        except Exception as e:
            print(f"Warning: Basic descriptors failed for molecule {i}: {e}")
            basic_descriptors = [0] * 10
        
        mol_features.extend(basic_descriptors)
        
        # Pad or truncate to exactly 200 features
        if len(mol_features) > 200:
            mol_features = mol_features[:200]
        elif len(mol_features) < 200:
            mol_features.extend([0] * (200 - len(mol_features)))
            
        features.append(mol_features)
    
    # Create feature names
    feature_names = [f'mol_feature_{i}' for i in range(200)]
    
    return np.array(features), feature_names

def create_dummy_features(smiles_list):
    """Create dummy features when RDKit is not available"""
    print("Creating dummy molecular features...")
    
    # Simple features based on SMILES string properties
    features = []
    for smiles in smiles_list:
        dummy_features = [
            len(smiles),  # Length of SMILES
            smiles.count('C'),  # Carbon count
            smiles.count('N'),  # Nitrogen count
            smiles.count('O'),  # Oxygen count
            smiles.count('S'),  # Sulfur count
            smiles.count('='),  # Double bonds
            smiles.count('#'),  # Triple bonds
            smiles.count('('),  # Branches
            smiles.count('['),  # Brackets
            smiles.count('c'),  # Aromatic carbons
        ]
        # Pad to 200 features with zeros
        dummy_features.extend([0] * (200 - len(dummy_features)))
        features.append(dummy_features)
    
    feature_names = [f'dummy_feature_{i}' for i in range(200)]
    return np.array(features), feature_names

# ===========================
# 3. MODELING
# ===========================

class MultiTargetRegressor:
    """Multi-target regressor for polymer properties"""
    
    def __init__(self, base_model=None):
        self.base_model = base_model or lgb.LGBMRegressor(
            random_state=42,
            n_estimators=500,
            learning_rate=0.05,
            max_depth=8,
            subsample=0.8,
            colsample_bytree=0.8,
            verbose=-1
        )
        self.models = {}
        self.properties = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
        
    def fit(self, X, y):
        """Fit separate models for each property"""
        print("Training models for each property...")
        
        for prop in self.properties:
            print(f"Training model for {prop}...")
            
            # Get non-null mask for this property
            mask = pd.notna(y[prop])
            
            if mask.sum() == 0:
                print(f"No valid data for {prop}, skipping...")
                continue
                
            X_prop = X[mask]
            y_prop = y[prop][mask]
            
            # Clone and fit model
            model = type(self.base_model)(**self.base_model.get_params())
            model.fit(X_prop, y_prop)
            
            self.models[prop] = model
            
            print(f"✓ {prop} model trained on {mask.sum()} samples")
    
    def predict(self, X):
        """Predict all properties"""
        predictions = {}
        
        for prop in self.properties:
            if prop in self.models:
                predictions[prop] = self.models[prop].predict(X)
            else:
                # Return zeros if no model was trained
                predictions[prop] = np.zeros(len(X))
        
        return pd.DataFrame(predictions)
    
    def get_feature_importance(self, top_n=20):
        """Get feature importance for each property"""
        importance_dict = {}
        
        for prop in self.properties:
            if prop in self.models:
                importance = self.models[prop].feature_importances_
                importance_dict[prop] = importance
        
        return importance_dict

# ===========================
# 4. EVALUATION
# ===========================

# Copy the WeightedMAEScorer from previous artifact
class WeightedMAEScorer:
    def __init__(self, property_ranges=None, sample_counts=None):
        if property_ranges is None:
            self.property_ranges = {
                'Tg': (-148.029738, 472.250000),
                'FFV': (0.226992, 0.777097),
                'Tc': (0.0465, 0.5240),
                'Density': (0.748691, 1.840999),
                'Rg': (9.728355, 34.672906)
            }
        else:
            self.property_ranges = property_ranges
            
        if sample_counts is None:
            self.sample_counts = {
                'Tg': 2538,
                'FFV': 9057,
                'Tc': 2764,
                'Density': 2640,
                'Rg': 2641
            }
        else:
            self.sample_counts = sample_counts
            
        self.weights = self._calculate_weights()
        
    def _calculate_weights(self):
        K = len(self.property_ranges)
        weights = {}
        
        sum_sqrt_inv_n = sum(np.sqrt(1/self.sample_counts[prop]) for prop in self.property_ranges.keys())
        
        for prop in self.property_ranges.keys():
            n_i = self.sample_counts[prop]
            r_i = self.property_ranges[prop][1] - self.property_ranges[prop][0]
            w_i = (1/r_i) * (K * np.sqrt(1/n_i) / sum_sqrt_inv_n)
            weights[prop] = w_i
            
        return weights
    
    def score(self, y_true, y_pred):
        if isinstance(y_true, pd.DataFrame):
            y_true = y_true.to_dict('series')
        if isinstance(y_pred, pd.DataFrame):
            y_pred = y_pred.to_dict('series')
            
        total_weighted_error = 0
        
        for prop in self.weights.keys():
            if prop in y_true and prop in y_pred:
                mask = pd.notna(y_true[prop]) & pd.notna(y_pred[prop])
                
                if mask.sum() > 0:
                    true_vals = y_true[prop][mask]
                    pred_vals = y_pred[prop][mask]
                    mae = np.mean(np.abs(true_vals - pred_vals))
                    weighted_error = self.weights[prop] * mae
                    total_weighted_error += weighted_error
        
        return total_weighted_error / len(self.weights)

# ===========================
# 5. MAIN PIPELINE
# ===========================

def run_complete_pipeline(train_path, test_path=None, submit=False):
    """Run the complete pipeline"""
    
    print("🚀 STARTING POLYMER PROPERTY PREDICTION PIPELINE")
    print("=" * 60)
    
    # 1. Load and explore data
    train_df, test_df = load_and_explore_data(train_path, test_path)
    
    # 2. Extract molecular features
    print("\n🧬 EXTRACTING MOLECULAR FEATURES")
    print("=" * 40)
    
    X_train, feature_names = extract_molecular_features(train_df['SMILES'].tolist())
    
    if test_df is not None:
        X_test, _ = extract_molecular_features(test_df['SMILES'].tolist())
    
    print(f"✓ Extracted {X_train.shape[1]} molecular features")
    
    # 3. Prepare target variables
    properties = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
    y_train = train_df[properties].copy()
    
    # 4. Initialize scorer
    scorer = WeightedMAEScorer()
    
    # 5. Train-validation split
    train_idx, val_idx = train_test_split(
        range(len(X_train)), 
        test_size=0.2, 
        random_state=42
    )
    
    X_train_fold = X_train[train_idx]
    X_val_fold = X_train[val_idx]
    y_train_fold = y_train.iloc[train_idx]
    y_val_fold = y_train.iloc[val_idx]
    
    # 6. Train model
    print("\n🤖 TRAINING MODELS")
    print("=" * 40)
    
    model = MultiTargetRegressor()
    model.fit(X_train_fold, y_train_fold)
    
    # 7. Validate
    print("\n📊 VALIDATION RESULTS")
    print("=" * 40)
    
    y_val_pred = model.predict(X_val_fold)
    val_score = scorer.score(y_val_fold, y_val_pred)
    
    print(f"Validation Weighted MAE: {val_score:.6f}")
    
    # Individual property scores
    print("\nIndividual Property MAE:")
    for prop in properties:
        mask = pd.notna(y_val_fold[prop])
        if mask.sum() > 0:
            y_true_prop = y_val_fold[prop].values[mask]
            y_pred_prop = y_val_pred[prop].values[mask]
            mae = mean_absolute_error(y_true_prop, y_pred_prop)
            print(f"{prop}: {mae:.4f}")
    
    # 8. Feature importance
    print("\n🔍 TOP FEATURE IMPORTANCES")
    print("=" * 40)
    
    importance_dict = model.get_feature_importance()
    for prop in properties:
        if prop in importance_dict:
            top_features = np.argsort(importance_dict[prop])[-5:][::-1]
            print(f"\n{prop} top features:")
            for i, feat_idx in enumerate(top_features):
                print(f"  {i+1}. Feature {feat_idx}: {importance_dict[prop][feat_idx]:.4f}")
    
    # 9. Train final model on all data
    print("\n🎯 TRAINING FINAL MODEL")
    print("=" * 40)
    
    final_model = MultiTargetRegressor()
    final_model.fit(X_train, y_train)
    
    # 10. Make predictions on test set
    if test_df is not None:
        print("\n🔮 MAKING TEST PREDICTIONS")
        print("=" * 40)
        
        test_predictions = final_model.predict(X_test)
        
        # Create submission
        submission = test_df[['id']].copy()
        for prop in properties:
            submission[prop] = test_predictions[prop]
        
        if submit:
            submission.to_csv('submission.csv', index=False)
            print("✅ Submission saved to submission.csv")
        
        return final_model, submission
    
    return final_model, None

# ===========================
# 6. USAGE EXAMPLE
# ===========================

if __name__ == "__main__":
    # Example usage - replace with your actual file paths
    
    # For Kaggle environment:
    # model, submission = run_complete_pipeline(
    #     '/kaggle/input/polymer-properties/train.csv',
    #     '/kaggle/input/polymer-properties/test.csv',
    #     submit=True
    # )
    
    # For local development:
    print("To use this pipeline:")
    print("1. Update the file paths below")
    print("2. Uncomment and run:")
    print()
    print("model, submission = run_complete_pipeline(")
    print("    'train.csv',")
    print("    'test.csv',")
    print("    submit=True")
    print(")")
    
    # Uncomment to run:

    model, submission = run_complete_pipeline(
        r'D:\Kaggle contests\neurips-open-polymer-prediction-2025\data\train.csv', 
        r'D:\Kaggle contests\neurips-open-polymer-prediction-2025\data\test.csv',
        submit=True
    )

To use this pipeline:
1. Update the file paths below
2. Uncomment and run:

model, submission = run_complete_pipeline(
    'train.csv',
    'test.csv',
    submit=True
)
🚀 STARTING POLYMER PROPERTY PREDICTION PIPELINE
=== LOADING DATA ===
Train shape: (7973, 7)
Test shape: (3, 2)

=== MISSING VALUES ===
         Missing  Percentage
Tg          7462   93.590869
Density     7360   92.311551
Rg          7359   92.299009
Tc          7236   90.756303
FFV          943   11.827418
id             0    0.000000
SMILES         0    0.000000

=== PROPERTY STATISTICS ===
               Tg          FFV          Tc     Density          Rg
count  511.000000  7030.000000  737.000000  613.000000  614.000000
mean    96.452314     0.367212    0.256334    0.985484   16.419787
std    111.228279     0.029609    0.089538    0.146189    4.608640
min   -148.029738     0.226992    0.046500    0.748691    9.728355
25%     13.674509     0.349549    0.186000    0.890243   12.540328
50%     74.040183     0.364264  

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
import warnings
warnings.filterwarnings('ignore')

# Install required packages (uncomment if needed)
# !pip install rdkit-pypi
# !pip install lightgbm
# !pip install xgboost
# !pip install catboost

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors, Crippen, Lipinski
    from rdkit.Chem.rdMolDescriptors import CalcNumRotatableBonds, CalcTPSA
    RDKIT_AVAILABLE = True
except ImportError:
    print("RDKit not available. Install with: pip install rdkit-pypi")
    RDKIT_AVAILABLE = False

import lightgbm as lgb
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    print("XGBoost not available")
    XGBOOST_AVAILABLE = False

try:
    import catboost as cb
    CATBOOST_AVAILABLE = True
except ImportError:
    print("CatBoost not available")
    CATBOOST_AVAILABLE = False

# ===========================
# 1. ENHANCED DATA LOADING & EDA
# ===========================

def load_and_explore_data(train_path, test_path=None):
    """Load data and perform comprehensive exploration"""
    print("=== LOADING DATA ===")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path) if test_path else None
    
    print(f"Train shape: {train_df.shape}")
    if test_df is not None:
        print(f"Test shape: {test_df.shape}")
    
    print("\n=== MISSING VALUES ===")
    properties = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
    missing = train_df[properties].isnull().sum()
    missing_pct = (missing / len(train_df)) * 100
    missing_df = pd.DataFrame({
        'Missing': missing,
        'Percentage': missing_pct
    }).sort_values('Missing', ascending=False)
    print(missing_df)
    
    # Enhanced property statistics with outlier detection
    print("\n=== PROPERTY STATISTICS ===")
    stats_df = train_df[properties].describe()
    print(stats_df)
    
    # Detect outliers using IQR method
    print("\n=== OUTLIER DETECTION ===")
    outlier_info = {}
    for prop in properties:
        if train_df[prop].notna().sum() > 0:
            Q1 = train_df[prop].quantile(0.25)
            Q3 = train_df[prop].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            outliers = train_df[(train_df[prop] < lower_bound) | (train_df[prop] > upper_bound)]
            outlier_info[prop] = len(outliers)
            print(f"{prop}: {len(outliers)} outliers ({len(outliers)/train_df[prop].notna().sum()*100:.1f}%)")
    
    return train_df, test_df, outlier_info

def visualize_data(train_df):
    """Create comprehensive visualizations for EDA"""
    properties = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
    
    # Distribution plots with box plots
    fig, axes = plt.subplots(3, 5, figsize=(20, 15))
    
    for i, prop in enumerate(properties):
        # Histogram
        train_df[prop].hist(bins=50, ax=axes[0, i], alpha=0.7)
        axes[0, i].set_title(f'{prop} Distribution')
        axes[0, i].set_xlabel(prop)
        axes[0, i].set_ylabel('Frequency')
        
        # Box plot
        train_df.boxplot(column=prop, ax=axes[1, i])
        axes[1, i].set_title(f'{prop} Box Plot')
        
        # Q-Q plot approximation
        from scipy import stats
        stats.probplot(train_df[prop].dropna(), dist="norm", plot=axes[2, i])
        axes[2, i].set_title(f'{prop} Q-Q Plot')
    
    plt.tight_layout()
    plt.show()
    
    # Enhanced correlation matrix with clustering
    plt.figure(figsize=(10, 8))
    correlation_matrix = train_df[properties].corr()
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
    sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
                square=True, linewidths=0.5, cbar_kws={"shrink": .8})
    plt.title('Property Correlations (Upper Triangle)')
    plt.show()
    
    return correlation_matrix

# ===========================
# 2. ENHANCED MOLECULAR FEATURE EXTRACTION
# ===========================

def extract_comprehensive_molecular_features(smiles_list):
    """Extract comprehensive molecular features from SMILES strings"""
    if not RDKIT_AVAILABLE:
        print("RDKit not available. Using enhanced dummy features.")
        return create_enhanced_dummy_features(smiles_list)
    
    features = []
    feature_names = []
    
    print("Extracting comprehensive molecular features...")
    
    # Define feature categories
    basic_descriptors = []
    topological_descriptors = []
    electronic_descriptors = []
    geometric_descriptors = []
    fingerprint_features = []
    
    for i, smiles in enumerate(smiles_list):
        if i % 1000 == 0:
            print(f"Processed {i}/{len(smiles_list)} molecules")
            
        mol = Chem.MolFromSmiles(smiles)
        
        if mol is None:
            # Handle invalid SMILES with more sophisticated fallback
            features.append([0] * 500)  # Increased feature dimension
            continue
        
        mol_features = []
        
        # 1. BASIC MOLECULAR DESCRIPTORS (20 features)
        try:
            basic_feats = [
                Descriptors.MolWt(mol),
                Descriptors.MolLogP(mol),
                Descriptors.NumHDonors(mol),
                Descriptors.NumHAcceptors(mol),
                Descriptors.NumRotatableBonds(mol),
                Descriptors.NumAromaticRings(mol),
                Descriptors.NumAliphaticRings(mol),
                Descriptors.RingCount(mol),
                mol.GetNumAtoms(),
                Descriptors.NumHeteroatoms(mol),
                mol.GetNumBonds(),
                mol.GetNumHeavyAtoms(),
                Descriptors.NumSaturatedRings(mol),
                Descriptors.NumAromaticCarbocycles(mol),
                Descriptors.NumAromaticHeterocycles(mol),
                Descriptors.NumAliphaticCarbocycles(mol),
                Descriptors.NumAliphaticHeterocycles(mol),
                Descriptors.FractionCsp3(mol),
                Descriptors.NumRadicalElectrons(mol),
                Descriptors.NumValenceElectrons(mol)
            ]
        except Exception as e:
            basic_feats = [0] * 20
        mol_features.extend(basic_feats)
        
        # 2. TOPOLOGICAL DESCRIPTORS (15 features)
        try:
            topo_feats = [
                Descriptors.BertzCT(mol),
                Descriptors.Ipc(mol),
                Descriptors.HallKierAlpha(mol),
                Descriptors.Kappa1(mol),
                Descriptors.Kappa2(mol),
                Descriptors.Kappa3(mol),
                Descriptors.Chi0(mol),
                Descriptors.Chi1(mol),
                Descriptors.Chi0n(mol),
                Descriptors.Chi1n(mol),
                Descriptors.Chi2n(mol),
                Descriptors.Chi3n(mol),
                Descriptors.Chi4n(mol),
                Descriptors.Chi0v(mol),
                Descriptors.Chi1v(mol)
            ]
        except Exception as e:
            topo_feats = [0] * 15
        mol_features.extend(topo_feats)
        
        # 3. ELECTRONIC DESCRIPTORS (10 features)
        try:
            elec_feats = [
                Descriptors.TPSA(mol),
                Descriptors.LabuteASA(mol),
                Descriptors.PEOE_VSA1(mol),
                Descriptors.PEOE_VSA2(mol),
                Descriptors.PEOE_VSA3(mol),
                Descriptors.SMR_VSA1(mol),
                Descriptors.SMR_VSA2(mol),
                Descriptors.SMR_VSA3(mol),
                Descriptors.SlogP_VSA1(mol),
                Descriptors.SlogP_VSA2(mol)
            ]
        except Exception as e:
            elec_feats = [0] * 10
        mol_features.extend(elec_feats)
        
        # 4. LIPINSKI AND DRUG-LIKE DESCRIPTORS (10 features)
        try:
            drug_feats = [
                Descriptors.qed(mol),
                Lipinski.NumHDonors(mol),
                Lipinski.NumHAcceptors(mol),
                Lipinski.NumRotatableBonds(mol),
                Descriptors.MolMR(mol),
                Descriptors.BalabanJ(mol),
                Descriptors.TPSA(mol),
                Crippen.MolLogP(mol),
                Crippen.MolMR(mol),
                len(Chem.GetSymmSSSR(mol))  # Number of rings
            ]
        except Exception as e:
            drug_feats = [0] * 10
        mol_features.extend(drug_feats)
        
        # 5. ENHANCED FINGERPRINTS (400+ features)
        # Morgan fingerprints with different radii
        try:
            morgan_2 = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, 2, nBits=128)
            morgan_3 = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, 3, nBits=128)
            mol_features.extend(list(morgan_2))
            mol_features.extend(list(morgan_3))
            
            # MACCS keys
            maccs = rdMolDescriptors.GetMACCSKeysFingerprint(mol)
            mol_features.extend(list(maccs))
            
            # Atom pair fingerprints
            ap_fp = rdMolDescriptors.GetAtomPairFingerprintAsBitVect(mol, nBits=64)
            mol_features.extend(list(ap_fp))
            
            # Topological torsion fingerprints
            tt_fp = rdMolDescriptors.GetTopologicalTorsionFingerprintAsBitVect(mol, nBits=64)
            mol_features.extend(list(tt_fp))
            
        except Exception as e:
            # Fallback fingerprints
            mol_features.extend([0] * (128 + 128 + 167 + 64 + 64))  # Total fingerprint features
        
        # Ensure exactly 500 features
        if len(mol_features) > 500:
            mol_features = mol_features[:500]
        elif len(mol_features) < 500:
            mol_features.extend([0] * (500 - len(mol_features)))
            
        features.append(mol_features)
    
    # Create comprehensive feature names
    feature_names = (
        [f'basic_{i}' for i in range(20)] +
        [f'topo_{i}' for i in range(15)] +
        [f'electronic_{i}' for i in range(10)] +
        [f'druglike_{i}' for i in range(10)] +
        [f'morgan2_{i}' for i in range(128)] +
        [f'morgan3_{i}' for i in range(128)] +
        [f'maccs_{i}' for i in range(167)] +
        [f'atompair_{i}' for i in range(64)] +
        [f'torsion_{i}' for i in range(64)] +
        [f'extra_{i}' for i in range(500 - 20 - 15 - 10 - 10 - 128 - 128 - 167 - 64 - 64)]
    )[:500]
    
    return np.array(features), feature_names

def create_enhanced_dummy_features(smiles_list):
    """Create enhanced dummy features when RDKit is not available"""
    print("Creating enhanced dummy molecular features...")
    
    features = []
    for smiles in smiles_list:
        # More sophisticated SMILES-based features
        dummy_features = [
            len(smiles),  # Length of SMILES
            smiles.count('C'),  # Carbon count
            smiles.count('N'),  # Nitrogen count
            smiles.count('O'),  # Oxygen count
            smiles.count('S'),  # Sulfur count
            smiles.count('P'),  # Phosphorus count
            smiles.count('F'),  # Fluorine count
            smiles.count('Cl'),  # Chlorine count
            smiles.count('Br'),  # Bromine count
            smiles.count('I'),  # Iodine count
            smiles.count('='),  # Double bonds
            smiles.count('#'),  # Triple bonds
            smiles.count('('),  # Branches
            smiles.count('['),  # Brackets
            smiles.count('c'),  # Aromatic carbons
            smiles.count('n'),  # Aromatic nitrogens
            smiles.count('o'),  # Aromatic oxygens
            smiles.count('s'),  # Aromatic sulfurs
            smiles.count('@'),  # Chirality
            smiles.count('%'),  # Ring closures
            # Ring counts
            smiles.count('1') + smiles.count('2') + smiles.count('3'),
            # Functional group approximations
            smiles.count('O=C'),  # Carbonyl groups
            smiles.count('N'),  # Nitrogen containing
            smiles.count('OH'),  # Hydroxyl groups
            smiles.count('CC'),  # Alkyl chains
        ]
        
        # Add more features based on SMILES patterns
        # Character frequency features
        for char in 'ABDEGHIJKLMRTUVWXYZabdefghijklmopqrtuvwxyz':
            dummy_features.append(smiles.count(char))
        
        # Pad to 500 features with statistical features
        while len(dummy_features) < 500:
            dummy_features.extend([
                len(smiles) % 7,  # Length mod operations
                hash(smiles) % 1000 / 1000,  # Normalized hash
                sum(ord(c) for c in smiles) / len(smiles) if smiles else 0,  # Average ASCII
            ])
            if len(dummy_features) >= 500:
                break
        
        # Truncate to exactly 500
        dummy_features = dummy_features[:500]
        features.append(dummy_features)
    
    feature_names = [f'dummy_feature_{i}' for i in range(500)]
    return np.array(features), feature_names

# ===========================
# 3. ADVANCED PREPROCESSING
# ===========================

class AdvancedPreprocessor:
    """Advanced preprocessing with multiple scaling and feature selection options"""
    
    def __init__(self, scaling_method='robust', feature_selection=True, n_features=300):
        self.scaling_method = scaling_method
        self.feature_selection = feature_selection
        self.n_features = n_features
        self.scalers = {}
        self.feature_selectors = {}
        self.selected_features = {}
        
    def fit_transform(self, X, y):
        """Fit preprocessor and transform training data"""
        properties = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
        X_processed = {}
        
        for prop in properties:
            print(f"Processing features for {prop}...")
            
            # Get mask for non-null values
            mask = pd.notna(y[prop])
            if mask.sum() == 0:
                continue
                
            X_prop = X[mask]
            y_prop = y[prop][mask]
            
            # 1. Scaling
            if self.scaling_method == 'standard':
                scaler = StandardScaler()
            elif self.scaling_method == 'robust':
                scaler = RobustScaler()
            elif self.scaling_method == 'quantile':
                scaler = QuantileTransformer(n_quantiles=min(1000, len(X_prop)))
            else:
                scaler = None
                
            if scaler:
                X_scaled = scaler.fit_transform(X_prop)
                self.scalers[prop] = scaler
            else:
                X_scaled = X_prop
            
            # 2. Feature Selection
            if self.feature_selection:
                # Use mutual information for feature selection
                selector = SelectKBest(mutual_info_regression, k=self.n_features)
                X_selected = selector.fit_transform(X_scaled, y_prop)
                self.feature_selectors[prop] = selector
                self.selected_features[prop] = selector.get_support()
            else:
                X_selected = X_scaled
                
            X_processed[prop] = X_selected
            
        return X_processed
    
    def transform(self, X):
        """Transform new data using fitted preprocessor"""
        X_processed = {}
        
        for prop in self.scalers.keys():
            # Apply scaling
            if prop in self.scalers:
                X_scaled = self.scalers[prop].transform(X)
            else:
                X_scaled = X
                
            # Apply feature selection
            if prop in self.feature_selectors:
                X_selected = self.feature_selectors[prop].transform(X_scaled)
            else:
                X_selected = X_scaled
                
            X_processed[prop] = X_selected
            
        return X_processed

# ===========================
# 4. ENHANCED MODELING WITH ENSEMBLE
# ===========================

class AdvancedMultiTargetRegressor:
    """Advanced multi-target regressor with ensemble methods"""
    
    def __init__(self, use_ensemble=True):
        self.use_ensemble = use_ensemble
        self.models = {}
        self.properties = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
        self.ensemble_weights = {}
        
    def _get_base_models(self):
        """Get base models for ensemble"""
        models = {}
        
        # LightGBM with optimized parameters
        models['lgb'] = lgb.LGBMRegressor(
            random_state=42,
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=10,
            num_leaves=100,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            min_child_samples=20,
            verbose=-1
        )
        
        # XGBoost
        if XGBOOST_AVAILABLE:
            models['xgb'] = xgb.XGBRegressor(
                random_state=42,
                n_estimators=1000,
                learning_rate=0.03,
                max_depth=8,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=0.1,
                verbosity=0
            )
        
        # CatBoost
        if CATBOOST_AVAILABLE:
            models['cat'] = cb.CatBoostRegressor(
                random_state=42,
                iterations=1000,
                learning_rate=0.03,
                depth=8,
                l2_leaf_reg=3,
                verbose=False
            )
        
        # Random Forest
        models['rf'] = RandomForestRegressor(
            random_state=42,
            n_estimators=500,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features='sqrt',
            n_jobs=-1
        )
        
        # Extra Trees
        models['et'] = ExtraTreesRegressor(
            random_state=42,
            n_estimators=500,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features='sqrt',
            n_jobs=-1
        )
        
        return models
    
    def fit(self, X_processed, y):
        """Fit ensemble models for each property"""
        print("Training advanced ensemble models...")
        
        for prop in self.properties:
            if prop not in X_processed:
                continue
                
            print(f"Training ensemble for {prop}...")
            
            X_prop = X_processed[prop]
            y_prop = y[prop][pd.notna(y[prop])]
            
            if self.use_ensemble:
                # Train multiple models
                base_models = self._get_base_models()
                property_models = {}
                
                # Split for ensemble training
                X_train, X_val, y_train, y_val = train_test_split(
                    X_prop, y_prop, test_size=0.2, random_state=42
                )
                
                # Train each base model
                predictions_val = {}
                for name, model in base_models.items():
                    print(f"  Training {name}...")
                    model.fit(X_train, y_train)
                    pred_val = model.predict(X_val)
                    predictions_val[name] = pred_val
                    property_models[name] = model
                
                # Calculate ensemble weights using validation performance
                weights = {}
                total_weight = 0
                for name, pred in predictions_val.items():
                    mae = mean_absolute_error(y_val, pred)
                    weight = 1 / (mae + 1e-8)  # Inverse MAE weighting
                    weights[name] = weight
                    total_weight += weight
                
                # Normalize weights
                for name in weights:
                    weights[name] /= total_weight
                
                self.models[prop] = property_models
                self.ensemble_weights[prop] = weights
                
                print(f"  Ensemble weights: {weights}")
                
            else:
                # Single best model (LightGBM)
                model = lgb.LGBMRegressor(
                    random_state=42,
                    n_estimators=1500,
                    learning_rate=0.02,
                    max_depth=12,
                    num_leaves=150,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    reg_alpha=0.1,
                    reg_lambda=0.1,
                    verbose=-1
                )
                model.fit(X_prop, y_prop)
                self.models[prop] = {'single': model}
                self.ensemble_weights[prop] = {'single': 1.0}
    
    def predict(self, X_processed):
        """Predict using ensemble"""
        predictions = {}
        
        for prop in self.properties:
            if prop not in self.models or prop not in X_processed:
                # Return zeros if no model
                predictions[prop] = np.zeros(len(list(X_processed.values())[0]))
                continue
            
            X_prop = X_processed[prop]
            
            if self.use_ensemble and len(self.models[prop]) > 1:
                # Ensemble prediction
                ensemble_pred = np.zeros(len(X_prop))
                
                for name, model in self.models[prop].items():
                    pred = model.predict(X_prop)
                    weight = self.ensemble_weights[prop][name]
                    ensemble_pred += weight * pred
                
                predictions[prop] = ensemble_pred
            else:
                # Single model prediction
                model = list(self.models[prop].values())[0]
                predictions[prop] = model.predict(X_prop)
        
        return pd.DataFrame(predictions)

# ===========================
# 5. ENHANCED EVALUATION WITH CV
# ===========================

class WeightedMAEScorer:
    def __init__(self, property_ranges=None, sample_counts=None):
        if property_ranges is None:
            self.property_ranges = {
                'Tg': (-148.029738, 472.250000),
                'FFV': (0.226992, 0.777097),
                'Tc': (0.0465, 0.5240),
                'Density': (0.748691, 1.840999),
                'Rg': (9.728355, 34.672906)
            }
        else:
            self.property_ranges = property_ranges
            
        if sample_counts is None:
            self.sample_counts = {
                'Tg': 2538,
                'FFV': 9057,
                'Tc': 2764,
                'Density': 2640,
                'Rg': 2641
            }
        else:
            self.sample_counts = sample_counts
            
        self.weights = self._calculate_weights()
        
    def _calculate_weights(self):
        K = len(self.property_ranges)
        weights = {}
        
        sum_sqrt_inv_n = sum(np.sqrt(1/self.sample_counts[prop]) for prop in self.property_ranges.keys())
        
        for prop in self.property_ranges.keys():
            n_i = self.sample_counts[prop]
            r_i = self.property_ranges[prop][1] - self.property_ranges[prop][0]
            w_i = (1/r_i) * (K * np.sqrt(1/n_i) / sum_sqrt_inv_n)
            weights[prop] = w_i
            
        return weights
    
    def score(self, y_true, y_pred):
        if isinstance(y_true, pd.DataFrame):
            y_true = y_true.to_dict('series')
        if isinstance(y_pred, pd.DataFrame):
            y_pred = y_pred.to_dict('series')
            
        total_weighted_error = 0
        
        for prop in self.weights.keys():
            if prop in y_true and prop in y_pred:
                mask = pd.notna(y_true[prop]) & pd.notna(y_pred[prop])
                
                if mask.sum() > 0:
                    true_vals = y_true[prop][mask]
                    pred_vals = y_pred[prop][mask]
                    mae = np.mean(np.abs(true_vals - pred_vals))
                    weighted_error = self.weights[prop] * mae
                    total_weighted_error += weighted_error
        
        return total_weighted_error / len(self.weights)

def cross_validate_model(X, y, model_class, preprocessor_class, n_folds=5, random_state=42):
    """Perform cross-validation with proper preprocessing"""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    scores = []
    scorer = WeightedMAEScorer()
    
    print(f"Performing {n_folds}-fold cross-validation...")
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        print(f"Fold {fold + 1}/{n_folds}")
        
        X_train_fold = X[train_idx]
        X_val_fold = X[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]
        
        # Fit preprocessor and model
        preprocessor = preprocessor_class()
        X_train_processed = preprocessor.fit_transform(X_train_fold, y_train_fold)
        X_val_processed = preprocessor.transform(X_val_fold)
        
        model = model_class()
        model.fit(X_train_processed, y_train_fold)
        
        y_pred = model.predict(X_val_processed)
        score = scorer.score(y_val_fold, y_pred)
        scores.append(score)
        
        print(f"  Fold {fold + 1} score: {score:.6f}")
    
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    print(f"CV Score: {mean_score:.6f} ± {std_score:.6f}")
    
    return scores, mean_score, std_score

# ===========================
# 6. ENHANCED MAIN PIPELINE
# ===========================

def run_enhanced_pipeline(train_path, test_path=None, submit=False, use_cv=True):
    """Run the enhanced pipeline with all optimizations"""
    
    print("🚀 STARTING ENHANCED POLYMER PROPERTY PREDICTION PIPELINE")
    print("=" * 70)
    
    # 1. Load and explore data
    train_df, test_df, outlier_info = load_and_explore_data(train_path, test_path)
    
    # 2. Extract comprehensive molecular features
    print("\n🧬 EXTRACTING COMPREHENSIVE MOLECULAR FEATURES")
    print("=" * 50)
    
    X_train, feature_names = extract_comprehensive_molecular_features(train_df['SMILES'].tolist())
    
    if test_df is not None:
        X_test, _ = extract_comprehensive_molecular_features(test_df['SMILES'].tolist())
    
    print(f"✅ Extracted {X_train.shape[1]} molecular features")
    
    # 3. Prepare target variables
    # 3. Prepare target variables
    properties = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
    y_train = train_df[properties].copy()
    
    print(f"✅ Target variables prepared: {properties}")
    
    # 4. Advanced preprocessing
    print("\n⚙️ ADVANCED PREPROCESSING")
    print("=" * 30)
    
    preprocessor = AdvancedPreprocessor(
        scaling_method='robust',
        feature_selection=True,
        n_features=300
    )
    
    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    
    if test_df is not None:
        X_test_processed = preprocessor.transform(X_test)
    
    print("✅ Advanced preprocessing completed")
    
    # 5. Model training with ensemble
    print("\n🤖 ADVANCED MODEL TRAINING")
    print("=" * 35)
    
    model = AdvancedMultiTargetRegressor(use_ensemble=True)
    model.fit(X_train_processed, y_train)
    
    print("✅ Ensemble model training completed")
    
    # 6. Cross-validation evaluation
    if use_cv:
        print("\n📊 CROSS-VALIDATION EVALUATION")
        print("=" * 40)
        
        cv_scores, mean_cv_score, std_cv_score = cross_validate_model(
            X_train, y_train, 
            lambda: AdvancedMultiTargetRegressor(use_ensemble=True),
            lambda: AdvancedPreprocessor(scaling_method='robust', feature_selection=True, n_features=300),
            n_folds=5
        )
        
        print(f"✅ Cross-validation completed: {mean_cv_score:.6f} ± {std_cv_score:.6f}")
    
    # 7. Generate predictions
    if test_df is not None:
        print("\n🔮 GENERATING PREDICTIONS")
        print("=" * 30)
        
        predictions = model.predict(X_test_processed)
        
        # Prepare submission
        submission = test_df[['id']].copy()
        for prop in properties:
            if prop in predictions.columns:
                submission[prop] = predictions[prop]
            else:
                submission[prop] = 0.0  # Default for missing predictions
        
        print("✅ Predictions generated")
        
        # Save submission
        if submit:
            submission_filename = 'submission.csv'
            submission.to_csv(submission_filename, index=False)
            print(f"✅ Submission saved to {submission_filename}")
        
        return model, preprocessor, submission
    
    return model, preprocessor, None

# ===========================
# 7. HYPERPARAMETER OPTIMIZATION
# ===========================

def optimize_hyperparameters(X_train, y_train, n_trials=50):
    """Optimize hyperparameters using Optuna (if available)"""
    try:
        import optuna
        
        def objective(trial):
            # LightGBM hyperparameters
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
                'max_depth': trial.suggest_int('max_depth', 6, 15),
                'num_leaves': trial.suggest_int('num_leaves', 50, 200),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 0.5),
                'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 0.5),
            }
            
            # Preprocessing parameters
            scaling_method = trial.suggest_categorical('scaling_method', ['standard', 'robust', 'quantile'])
            n_features = trial.suggest_int('n_features', 200, 500)
            
            # Create preprocessor
            preprocessor = AdvancedPreprocessor(
                scaling_method=scaling_method,
                feature_selection=True,
                n_features=n_features
            )
            
            # Cross-validation
            kf = KFold(n_splits=3, shuffle=True, random_state=42)
            scores = []
            scorer = WeightedMAEScorer()
            
            for train_idx, val_idx in kf.split(X_train):
                X_train_fold = X_train[train_idx]
                X_val_fold = X_train[val_idx]
                y_train_fold = y_train.iloc[train_idx]
                y_val_fold = y_train.iloc[val_idx]
                
                # Preprocess
                X_train_processed = preprocessor.fit_transform(X_train_fold, y_train_fold)
                X_val_processed = preprocessor.transform(X_val_fold)
                
                # Train models for each property
                fold_predictions = {}
                for prop in ['Tg', 'FFV', 'Tc', 'Density', 'Rg']:
                    if prop in X_train_processed:
                        model = lgb.LGBMRegressor(**params, random_state=42, verbose=-1)
                        model.fit(X_train_processed[prop], y_train_fold[prop].dropna())
                        
                        pred = model.predict(X_val_processed[prop])
                        fold_predictions[prop] = pred
                
                # Calculate score
                score = scorer.score(y_val_fold, pd.DataFrame(fold_predictions))
                scores.append(score)
            
            return np.mean(scores)
        
        study = optuna.create_study(direction='minimize')
        study.optimize(objective, n_trials=n_trials)
        
        print(f"Best parameters: {study.best_params}")
        print(f"Best score: {study.best_value}")
        
        return study.best_params
        
    except ImportError:
        print("Optuna not available for hyperparameter optimization")
        return None

# ===========================
# 8. MAIN EXECUTION
# ===========================

if __name__ == "__main__":
    # File paths
    TRAIN_PATH = "train.csv"
    TEST_PATH = "test.csv"
    
    # Run the enhanced pipeline
    try:
        model, preprocessor, submission = run_enhanced_pipeline(
            TRAIN_PATH, 
            TEST_PATH, 
            submit=True,
            use_cv=True
        )
        
        print("\n🎉 PIPELINE COMPLETED SUCCESSFULLY!")
        print("=" * 50)
        
        if submission is not None:
            print(f"📄 Submission shape: {submission.shape}")
            print(f"📊 Submission preview:")
            print(submission.head())
        
    except Exception as e:
        print(f"❌ Pipeline failed with error: {str(e)}")
        import traceback
        traceback.print_exc()

# ===========================
# 9. ADDITIONAL UTILITIES
# ===========================

def analyze_feature_importance(model, feature_names, top_k=20):
    """Analyze feature importance across all properties"""
    print("\n📊 FEATURE IMPORTANCE ANALYSIS")
    print("=" * 40)
    
    importance_data = {}
    
    for prop in model.properties:
        if prop in model.models:
            prop_models = model.models[prop]
            
            # Get feature importance from LightGBM model
            if 'lgb' in prop_models:
                lgb_model = prop_models['lgb']
                importance = lgb_model.feature_importances_
                
                # Get top features
                top_indices = np.argsort(importance)[-top_k:][::-1]
                
                importance_data[prop] = {
                    'features': [feature_names[i] for i in top_indices],
                    'importance': importance[top_indices]
                }
                
                print(f"\n{prop} - Top {top_k} Important Features:")
                for i, (feat, imp) in enumerate(zip(importance_data[prop]['features'], 
                                                   importance_data[prop]['importance'])):
                    print(f"  {i+1:2d}. {feat}: {imp:.4f}")
    
    return importance_data

def generate_prediction_intervals(model, X_processed, confidence=0.95):
    """Generate prediction intervals using bootstrap"""
    print(f"\n🔍 GENERATING {confidence*100}% PREDICTION INTERVALS")
    print("=" * 50)
    
    from sklearn.utils import resample
    
    predictions = {}
    intervals = {}
    
    for prop in model.properties:
        if prop not in X_processed or prop not in model.models:
            continue
            
        X_prop = X_processed[prop]
        prop_models = model.models[prop]
        
        # Generate bootstrap predictions
        bootstrap_preds = []
        
        for _ in range(100):  # 100 bootstrap samples
            if model.use_ensemble:
                ensemble_pred = np.zeros(len(X_prop))
                for name, mdl in prop_models.items():
                    pred = mdl.predict(X_prop)
                    weight = model.ensemble_weights[prop][name]
                    ensemble_pred += weight * pred
                bootstrap_preds.append(ensemble_pred)
            else:
                mdl = list(prop_models.values())[0]
                pred = mdl.predict(X_prop)
                bootstrap_preds.append(pred)
        
        bootstrap_preds = np.array(bootstrap_preds)
        
        # Calculate prediction intervals
        alpha = 1 - confidence
        lower_percentile = (alpha/2) * 100
        upper_percentile = (1 - alpha/2) * 100
        
        pred_mean = np.mean(bootstrap_preds, axis=0)
        pred_lower = np.percentile(bootstrap_preds, lower_percentile, axis=0)
        pred_upper = np.percentile(bootstrap_preds, upper_percentile, axis=0)
        
        predictions[prop] = pred_mean
        intervals[prop] = {
            'lower': pred_lower,
            'upper': pred_upper,
            'width': pred_upper - pred_lower
        }
        
        print(f"{prop} - Average interval width: {np.mean(intervals[prop]['width']):.4f}")
    
    return predictions, intervals

def save_model_artifacts(model, preprocessor, feature_names, filepath_prefix="model"):
    """Save model artifacts for later use"""
    import pickle
    
    artifacts = {
        'model': model,
        'preprocessor': preprocessor,
        'feature_names': feature_names,
        'properties': model.properties,
        'timestamp': pd.Timestamp.now().isoformat()
    }
    
    with open(f"{filepath_prefix}_artifacts.pkl", 'wb') as f:
        pickle.dump(artifacts, f)
    
    print(f"✅ Model artifacts saved to {filepath_prefix}_artifacts.pkl")

def load_model_artifacts(filepath):
    """Load saved model artifacts"""
    import pickle
    
    with open(filepath, 'rb') as f:
        artifacts = pickle.load(f)
    
    print(f"✅ Model artifacts loaded from {filepath}")
    return artifacts

XGBoost not available
CatBoost not available
🚀 STARTING ENHANCED POLYMER PROPERTY PREDICTION PIPELINE
=== LOADING DATA ===
Train shape: (7973, 7)
Test shape: (3, 2)

=== MISSING VALUES ===
         Missing  Percentage
Tg          7462   93.590869
Density     7360   92.311551
Rg          7359   92.299009
Tc          7236   90.756303
FFV          943   11.827418

=== PROPERTY STATISTICS ===
               Tg          FFV          Tc     Density          Rg
count  511.000000  7030.000000  737.000000  613.000000  614.000000
mean    96.452314     0.367212    0.256334    0.985484   16.419787
std    111.228279     0.029609    0.089538    0.146189    4.608640
min   -148.029738     0.226992    0.046500    0.748691    9.728355
25%     13.674509     0.349549    0.186000    0.890243   12.540328
50%     74.040183     0.364264    0.236000    0.948193   15.052194
75%    161.147595     0.380790    0.330500    1.062096   20.411067
max    472.250000     0.777097    0.524000    1.840999   34.672906

=== 